In [113]:
import deepchem as dc
from deepchem.models import GCNModel
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from rdkit import Chem
from sklearn.model_selection import train_test_split
import torch

In [114]:

def is_valid_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        return mol is not None and mol.GetNumAtoms() > 1
    except Exception:
        return False

def load_single_task_data(target_col):
    """加载单个任务的数据，并仅保留目标值为 0 和 1 的样本"""
    df = pd.read_csv('tox21_cleaned.csv')
    
    # 过滤当前任务的缺失值
    df = df.dropna(subset=[target_col])
    print(f"\nTask {target_col} Valid sample size: {len(df)}")
       
 # 过滤有效的 SMILES 并保留索引
    valid_smiles_indices = df['smiles'].apply(is_valid_smiles)
    
    # 过滤目标列值为 0 或 1 的样本
    valid_target_indices = df[target_col].isin([0, 1])
    
    # 合并两个条件，得到最终有效索引
    valid_indices = valid_smiles_indices & valid_target_indices
    valid_df = df[valid_indices]
    
    # 提取有效的 SMILES 和目标值
    valid_smiles = valid_df['smiles'].values
    y = valid_df[target_col].values.astype(np.float32)
    
    return valid_smiles, y

In [115]:
def  train_single_task(target_col):

    data = load_single_task_data(target_col)
    smiles = data[0]
    y = data[1]
    
    featurizer = dc.feat.MolGraphConvFeaturizer()
   
    
    # 划分训练集和测试集
    train_smiles, test_smiles, train_y, test_y = train_test_split(smiles, y, test_size=0.3, random_state=4)
    
    # 创建图数据集
    train_dataset = dc.data.NumpyDataset(X=featurizer.featurize(train_smiles), y= train_y)
    test_dataset = dc.data.NumpyDataset(X=featurizer.featurize(test_smiles), y=test_y )
    
    # 初始化模型
    model = GCNModel(
        n_tasks=1, 
        mode='classification',  
        batch_size=256,
        learning_rate=0.001,
        weight_decay=0.001,
        graph_conv_layers=[128,128,128],
    )
    
    
    # 训练模型
    model.fit(train_dataset, nb_epoch=20)
    

    # 评估指标
    metrics = [
        dc.metrics.Metric(dc.metrics.roc_auc_score, name="AUC"),
        dc.metrics.Metric(dc.metrics.f1_score, name="F1"),
        dc.metrics.Metric(dc.metrics.recall_score, name="Recall"),
        dc.metrics.Metric(dc.metrics.accuracy_score, name="Accuracy")
    ]

    # 生成DataFrame
    results = model.evaluate(test_dataset, metrics)
    performance_df = pd.DataFrame([results], columns=results.keys())

    print(performance_df.round(4))
    
    return performance_df

In [116]:

TARGET_COLS = ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 
              'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE',
              'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']

if __name__ == "__main__":
    results = []
    for target in TARGET_COLS:
        task_df = train_single_task(target)
        task_df.insert(0, 'Task', target)  
        results.append(task_df.round(4))

# 合并所有结果
final_report = pd.concat(results, ignore_index=True)

# 格式化输出
print(final_report.to_string(index=False))


Task NR-AR Valid sample size: 7432
      AUC      F1  Recall  Accuracy
0  0.7825  0.4817  0.4842    0.9555

Task NR-AR-LBD Valid sample size: 6895
      AUC      F1  Recall  Accuracy
0  0.8244  0.5591  0.4127    0.9801

Task NR-AhR Valid sample size: 6684
      AUC      F1  Recall  Accuracy
0  0.8912  0.4134  0.2753    0.9035

Task NR-Aromatase Valid sample size: 5934
      AUC      F1  Recall  Accuracy
0  0.8564  0.2581  0.2128    0.9352

Task NR-ER Valid sample size: 6309
      AUC      F1  Recall  Accuracy
0  0.6923  0.1993  0.1261    0.8724

Task NR-ER-LBD Valid sample size: 7105
      AUC      F1  Recall  Accuracy
0  0.7129  0.0196  0.0099     0.953

Task NR-PPAR-gamma Valid sample size: 6576
      AUC      F1  Recall  Accuracy
0  0.7576  0.0656  0.0339     0.971

Task SR-ARE Valid sample size: 5928
      AUC      F1  Recall  Accuracy
0  0.7356  0.1808  0.1142    0.8315

Task SR-ATAD5 Valid sample size: 7225
      AUC      F1  Recall  Accuracy
0  0.8354  0.1111  0.0845    0.9556


In [117]:
# 合并所有结果
final_report = pd.concat(results, ignore_index=True)

# 计算每一列的平均值（忽略第一列）
mean_values = final_report.iloc[:, 1:].mean()

# 将平均值添加为新行，并指定名称为 'overall'
final_report.loc[len(final_report)] = ['overall'] + mean_values.tolist()

# 打印结果
display(final_report)

,Task,AUC,F1,Recall,Accuracy
0,NR-AR,0.7825,0.4817,0.484200,0.955500
1,NR-AR-LBD,0.8244,0.5591,0.412700,0.980100
2,NR-AhR,0.8912,0.4134,0.275300,0.903500
3,NR-Aromatase,0.8564,0.2581,0.212800,0.935200
4,NR-ER,0.6923,0.1993,0.126100,0.872400
5,NR-ER-LBD,0.7129,0.0196,0.009900,0.953000
6,NR-PPAR-gamma,0.7576,0.0656,0.033900,0.971000
7,SR-ARE,0.7356,0.1808,0.114200,0.831500
8,SR-ATAD5,0.8354,0.1111,0.084500,0.955600
9,SR-HSE,0.7812,0.2449,0.166700,0.943700


In [118]:
#小数据集测试

data = load_single_task_data('NR-AR')
smiles = data[0][:1500]
y = data[1][:1500]

tsmiles = data[0][1000:]
ty = data[1][1000:]

featurizer = dc.feat.MolGraphConvFeaturizer()
dataset = dc.data.NumpyDataset(X=featurizer.featurize(smiles), y=y)
tdataset = dc.data.NumpyDataset(X=featurizer.featurize(tsmiles), y=ty)



# 初始化模型
# model = GCNModel(
#     n_tasks=1, 
#     mode='classification',  
#     batch_size=128,
#     learning_rate=0.001,
# )
# 0.83

model = GCNModel(
        n_tasks=1, 
        mode='classification',  
        batch_size=256,
        learning_rate=0.001,
        dropout=0.1,


) 

# model = GCNModel(
#     n_tasks=1, 
#     mode='classification',  
#     batch_size=256,
#     learning_rate=0.0001,
#     dropout=0.5,
#     weight_decay=0.0001,
# ) 0.91
    
    
# model = GCNModel(mode='classification', n_tasks=1)
model.fit(dataset, nb_epoch=20)

# 评估指标
metrics = [
    dc.metrics.Metric(dc.metrics.roc_auc_score, name="AUC"),
    dc.metrics.Metric(dc.metrics.f1_score, name="F1"),
    dc.metrics.Metric(dc.metrics.recall_score, name="Recall"),
    dc.metrics.Metric(dc.metrics.accuracy_score, name="Accuracy")
]

# 生成DataFrame
results = model.evaluate(tdataset, metrics)
performance_df = pd.DataFrame([results], columns=results.keys())

print(performance_df.round(4))


Task NR-AR Valid sample size: 7432
      AUC      F1  Recall  Accuracy
0  0.7802  0.4345  0.2836    0.9683
